---
## checking where object '0711' is

# -> DONE wasn't part of v1 (is part of v2)

In [2]:
import os
os.chdir("../../web_backend/")

In [3]:
from tqdm import tqdm
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns

from data.data import CollectionAccessor, ImageHandler, EmbeddingSpaceAccessor

from search import Search, GraphSearcher, TextEmbeddingSearcher, EmbeddingSearcher

In [4]:
from app import init_DMG, search_collection

In [5]:
def init_DMG():
    DMG_DIR = "./data/DMG"
    image_folder = DMG_DIR+"/images/"
    image_handler = ImageHandler("DMG", image_folder=image_folder, keep_prefix=False)

    time_stamp, pub_file, priv_file = CollectionAccessor.get_latest_dump(DMG_DIR+"/dumps")
    print(time_stamp)

    dmg_meta = dict(name="Design Museum Gent (public & private)", id_="DMG_"+time_stamp,
                creation_timestamp=time_stamp, language="nl")
    df = CollectionAccessor.get_DMG(pub_path=pub_file, #get_latest("./data/dumps", contains="public"),
                                     priv_path=priv_file, #get_latest("./data/dumps", contains="private"),
                                     rights_path=DMG_DIR+"/rights.csv",
                                     image_handler=image_handler,
                                     **dmg_meta)

    kg_searcher = GraphSearcher(df)


    sem_embs = EmbeddingSpaceAccessor.load(DMG_DIR+"/generated_data/distiluse-base-multilingual-cased-v2",
                                       loadXD=None)
    concept_search = TextEmbeddingSearcher(sem_embs, name="concept-searcher")


    sem_embs = EmbeddingSpaceAccessor.load(DMG_DIR+"/generated_data/distiluse-base-multilingual-cased-v2",
                                       loadXD=32)
    sem_searcher = EmbeddingSearcher(sem_embs, name="semantic-searcher")
    
    viz_embs = EmbeddingSpaceAccessor.load(DMG_DIR+"/generated_data/vitmae", loadXD=32)
    viz_searcher = EmbeddingSearcher(viz_embs, name="visual-searcher")

    s = Search([kg_searcher, sem_searcher, viz_searcher])
    return df, s, concept_search, sem_embs, sem_searcher

df, s, cs, sem_embs, sem_searcher = init_DMG()

2026-03-28


[GraphSearcher]: building graph...: 100%|████████████████████████████| 21030/21030 [00:02<00:00, 8515.90it/s]


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

In [8]:
df.index.str.startswith("0711").sum()

np.int64(0)

In [20]:
pub = pd.read_csv("./data/DMG/dumps/API_dump_public_2026-03-28_extracted.csv")
priv = pd.read_csv("./data/DMG/dumps/API_dump_private_2026-03-28_extracted.csv")
df = pd.concat([pub, priv], axis=0).set_index("object_number")

In [22]:
df.index.str.startswith("0711").sum()

np.int64(0)

---
#### in the JSON?

In [24]:
import json

In [43]:
with open("./data/DMG/dumps/API_dump_public_2026-03-28.json") as handle:
    pub = json.load(handle)

with open("./data/DMG/dumps/API_dump_private_2026-03-28.json") as handle:
    priv = json.load(handle)

def get_obj_num(rec):
    if len(rec) < 1:
        return None
    second = rec['http://www.w3.org/ns/adms#identifier'][1]
    try:
        return second['http://www.w3.org/2004/02/skos/core#notation']["@value"]
    except KeyError:
        return None
    # prefix = 'https://data.designmuseumgent.be/v2/id/object/'
    # return rec['@id'].replace(prefix, "")

obj_nums = pd.Series(list(map(get_obj_num, pub+priv))).dropna()

In [49]:
# obj_nums.astype(str).index.str.startswith("0711").sum()

[o for o in obj_nums if str(o).startswith("0711")]

[]

---
# swapping objects in `/search/order`

In [11]:
k = 3
df = pd.DataFrame(list(range(100, 120, 2)), index=list(range(0, 20, 2)))

rand = df.sample(k)

rand_idx = df.index.isin(rand.index).nonzero()[0]

In [9]:
rand

,0
18,118
0,100
12,112


In [10]:
test

,0
0,100
2,102
4,104
6,106
8,108
10,110
12,112
14,114
16,116
18,118


In [7]:
orig = test.iloc[:k]

test.loc[rand.index] = orig.copy().values
test.loc[orig.index] = rand.copy().values

In [60]:
test.iloc[rand_idx] = orig
test.iloc[:k] = rand

In [22]:
rand = df.sample(k)
pos_i = [df.index.get_loc(r) for r in rand.index]
pos_j = [df.index.get_loc(i) for i in df.index[:k]]

order = np.arange(len(df))
order[pos_i], order[pos_j] = order[pos_j], order[pos_i]

df = df.iloc[order]

In [24]:
df

,0
2,102
16,116
10,110
6,106
8,108
4,104
0,100
14,114
12,112
18,118


In [25]:
order

array([6, 8, 5, 3, 4, 2, 0, 7, 1, 9])

In [ ]:
http://127.0.0.1:8080/DMG_2026-08-01/search/sample?object_ids=FH-0066_1-6%2C0982&k=12